# Exercise 6 - Text Similarity

**Anggota Kelompok:**
1. M. Furqon Lutfi Septriandri (25031554192)
2. Akram Ali Fi Natarik (25031554212)

**Soal:** terapkan *text similarity* pada teks hasil *feature engineering* dari Exercise 5, lalu analisis hasilnya.

**Data (hasil pre-processing Exercise 4, sama seperti Exercise 5):**
1. `detik_preprocessed.csv` - artikel detik.com (Bahasa Indonesia, 58 dokumen, rata-rata sekitar 4 kata per dokumen)
2. `jsonplaceholder_preprocessed.csv` - JSONPlaceholder `/posts` (teks dummy *lorem ipsum*, 100 dokumen, rata-rata sekitar 30 kata per dokumen)

**Alur (mengikuti materi Chapter 7):**
1. Cosine Similarity (pada TF-IDF dan Bag-of-Words hasil feature engineering)
2. Jaccard Similarity (pada One-Hot / binary)
3. Pencarian dokumen berdasarkan query (Top-K), seperti contoh "Data Real" di materi
4. Cosine Similarity pada vektor Word2Vec & FastText
5. Perbandingan & kesimpulan

## 0. Import Library

In [ ]:
import warnings
warnings.filterwarnings(action='ignore')

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import pdist, cdist, squareform

import nltk
from nltk import word_tokenize
from tqdm import tqdm
import gensim.models

nltk.download('punkt')
nltk.download('punkt_tab')

## 1. Load Data Hasil Pre-processing

In [ ]:
df_detik = pd.read_csv("detik_preprocessed.csv")
df_detik['prepro'] = df_detik['prepro'].fillna('')
df_detik.shape

In [ ]:
df_api = pd.read_csv("jsonplaceholder_preprocessed.csv")
df_api['prepro'] = df_api['prepro'].fillna('')
df_api.shape

## 2. Feature Engineering (dari Exercise 5)
Matriks fitur dibangun ulang dengan cara yang sama seperti Exercise 5: **Bag-of-Words (count)**, **One-Hot (binary)**, dan **TF-IDF**.

### 2.1 Dataset detik.com

In [ ]:
vec_count_detik = CountVectorizer()
X_count_detik = vec_count_detik.fit_transform(df_detik['prepro'])

vec_bin_detik = CountVectorizer(binary=True)
X_bin_detik = vec_bin_detik.fit_transform(df_detik['prepro']).toarray()

vec_tfidf_detik = TfidfVectorizer()
X_tfidf_detik = vec_tfidf_detik.fit_transform(df_detik['prepro'])

X_count_detik.shape, X_bin_detik.shape, X_tfidf_detik.shape

### 2.2 Dataset JSONPlaceholder

In [ ]:
vec_count_api = CountVectorizer()
X_count_api = vec_count_api.fit_transform(df_api['prepro'])

vec_bin_api = CountVectorizer(binary=True)
X_bin_api = vec_bin_api.fit_transform(df_api['prepro']).toarray()

vec_tfidf_api = TfidfVectorizer()
X_tfidf_api = vec_tfidf_api.fit_transform(df_api['prepro'])

X_count_api.shape, X_bin_api.shape, X_tfidf_api.shape

## 3. Cosine Similarity

### 3.1 Cosine pada TF-IDF - dataset detik.com

In [ ]:
sim_tfidf_detik = cosine_similarity(X_tfidf_detik)
pd.DataFrame(sim_tfidf_detik).round(2).iloc[:8, :8]

In [ ]:
iu = np.triu_indices_from(sim_tfidf_detik, k=1)
pairs_tfidf_detik = pd.DataFrame({
    'doc_1': iu[0],
    'doc_2': iu[1],
    'similarity': sim_tfidf_detik[iu]
})
pairs_tfidf_detik['teks_1'] = df_detik['prepro'].iloc[pairs_tfidf_detik['doc_1']].values
pairs_tfidf_detik['teks_2'] = df_detik['prepro'].iloc[pairs_tfidf_detik['doc_2']].values
pairs_tfidf_detik.sort_values('similarity', ascending=False).head(5)

In [ ]:
vals = sim_tfidf_detik[iu]
print("Rata-rata similarity      :", round(vals.mean(), 3))
print("Similarity maksimum       :", round(vals.max(), 3))
print("Persentase pasangan nol   :", round((vals == 0).mean() * 100, 1), "%")

### 3.2 Cosine pada TF-IDF - dataset JSONPlaceholder

In [ ]:
sim_tfidf_api = cosine_similarity(X_tfidf_api)
iu_api = np.triu_indices_from(sim_tfidf_api, k=1)
pairs_tfidf_api = pd.DataFrame({
    'doc_1': iu_api[0],
    'doc_2': iu_api[1],
    'similarity': sim_tfidf_api[iu_api]
})
pairs_tfidf_api['teks_1'] = df_api['prepro'].iloc[pairs_tfidf_api['doc_1']].str[:60].values
pairs_tfidf_api['teks_2'] = df_api['prepro'].iloc[pairs_tfidf_api['doc_2']].str[:60].values
pairs_tfidf_api.sort_values('similarity', ascending=False).head(5)

In [ ]:
vals_api = sim_tfidf_api[iu_api]
print("Rata-rata similarity      :", round(vals_api.mean(), 3))
print("Similarity maksimum       :", round(vals_api.max(), 3))
print("Persentase pasangan nol   :", round((vals_api == 0).mean() * 100, 1), "%")

### 3.3 Cosine pada Bag-of-Words (count) vs TF-IDF

In [ ]:
sim_count_detik = cosine_similarity(X_count_detik)
sim_count_api = cosine_similarity(X_count_api)

pd.DataFrame({
    'Dataset': ['detik', 'jsonplaceholder'],
    'Rata-rata (BoW count)': [sim_count_detik[iu].mean(), sim_count_api[iu_api].mean()],
    'Rata-rata (TF-IDF)': [sim_tfidf_detik[iu].mean(), sim_tfidf_api[iu_api].mean()],
    'Maks (BoW count)': [sim_count_detik[iu].max(), sim_count_api[iu_api].max()],
    'Maks (TF-IDF)': [sim_tfidf_detik[iu].max(), sim_tfidf_api[iu_api].max()],
}).round(3)

### Analisis Cosine Similarity

- **detik.com:** dokumen sangat pendek (rata-rata sekitar 4 kata hasil pre-processing), sehingga **sekitar 90% pasangan dokumen punya cosine = 0** (tidak berbagi satu kata pun). Rata-rata similarity hanya sekitar 0.02, dengan nilai tertinggi sekitar 0.69 pada pasangan yang sama-sama memuat kata `laku` (`laku orang laku` dan `laku laku sepenuh hati`). Jadi matriks similarity-nya sangat *sparse*: kemiripan hanya muncul kalau ada kata kunci yang sama persis.
- **JSONPlaceholder:** dokumen lebih panjang (rata-rata sekitar 30 kata) dan kosakatanya sempit (hanya sekitar 181 kata unik untuk 100 dokumen), sehingga **hampir tidak ada pasangan dengan cosine nol**. Rata-rata similarity sekitar 0.155, tetapi nilai tertinggi cuma sekitar 0.43. Artinya semua dokumen "agak mirip" satu sama lain dan tidak ada yang benar-benar menonjol, wajar karena isinya teks *lorem ipsum* acak yang memakai kata-kata filler yang sama berulang-ulang.
- **TF-IDF vs BoW count:** TF-IDF menurunkan bobot kata yang muncul di banyak dokumen, jadi nilai similarity pada kata-kata umum tidak lagi mendominasi. Terlihat pada tabel 3.3: rata-rata similarity pada jsonplaceholder turun dari sekitar 0.22 (BoW count) menjadi sekitar 0.155 (TF-IDF), dan nilai maksimumnya dari sekitar 0.61 menjadi 0.43, karena kata-kata filler yang muncul di hampir semua dokumen tidak lagi menggelembungkan kemiripan. Pada detik selisihnya kecil (0.026 menjadi 0.019) karena dokumennya terlalu pendek untuk punya banyak kata umum.

## 4. Jaccard Similarity (pada One-Hot / binary)

### 4.1 Dataset detik.com

In [ ]:
dist_jaccard_detik = pdist(X_bin_detik, metric='jaccard')
sim_jaccard_detik = 1 - squareform(dist_jaccard_detik)

pairs_jaccard_detik = pd.DataFrame({
    'doc_1': iu[0],
    'doc_2': iu[1],
    'similarity': sim_jaccard_detik[iu]
})
pairs_jaccard_detik['teks_1'] = df_detik['prepro'].iloc[pairs_jaccard_detik['doc_1']].values
pairs_jaccard_detik['teks_2'] = df_detik['prepro'].iloc[pairs_jaccard_detik['doc_2']].values
pairs_jaccard_detik.sort_values('similarity', ascending=False).head(5)

### 4.2 Dataset JSONPlaceholder

In [ ]:
dist_jaccard_api = pdist(X_bin_api, metric='jaccard')
sim_jaccard_api = 1 - squareform(dist_jaccard_api)

pairs_jaccard_api = pd.DataFrame({
    'doc_1': iu_api[0],
    'doc_2': iu_api[1],
    'similarity': sim_jaccard_api[iu_api]
})
pairs_jaccard_api['teks_1'] = df_api['prepro'].iloc[pairs_jaccard_api['doc_1']].str[:60].values
pairs_jaccard_api['teks_2'] = df_api['prepro'].iloc[pairs_jaccard_api['doc_2']].str[:60].values
pairs_jaccard_api.sort_values('similarity', ascending=False).head(5)

### 4.3 Cosine (TF-IDF) vs Jaccard

In [ ]:
korelasi = pd.DataFrame({
    'Dataset': ['detik', 'jsonplaceholder'],
    'Rata-rata Cosine TF-IDF': [sim_tfidf_detik[iu].mean(), sim_tfidf_api[iu_api].mean()],
    'Rata-rata Jaccard': [sim_jaccard_detik[iu].mean(), sim_jaccard_api[iu_api].mean()],
    'Maks Cosine TF-IDF': [sim_tfidf_detik[iu].max(), sim_tfidf_api[iu_api].max()],
    'Maks Jaccard': [sim_jaccard_detik[iu].max(), sim_jaccard_api[iu_api].max()],
    'Korelasi Cosine vs Jaccard': [
        np.corrcoef(sim_tfidf_detik[iu], sim_jaccard_detik[iu])[0, 1],
        np.corrcoef(sim_tfidf_api[iu_api], sim_jaccard_api[iu_api])[0, 1]],
}).round(3)
korelasi

### Analisis Jaccard Similarity

- Jaccard hanya melihat **ada/tidaknya kata** (irisan dibagi gabungan himpunan kata), tidak peduli berapa kali sebuah kata muncul dan tidak memberi bobot ke kata langka. Karena itu nilainya selalu lebih kecil atau sama dengan cosine pada dokumen yang sama: pada detik rata-rata sekitar 0.015 (maks 0.40), pada jsonplaceholder rata-rata sekitar 0.113 (maks 0.30).
- **Korelasi antara cosine TF-IDF dan Jaccard tinggi**: sekitar 0.94 pada detik dan 0.87 pada jsonplaceholder. Artinya kedua metode umumnya sepakat soal pasangan mana yang mirip, hanya skalanya yang berbeda. Korelasi di detik lebih tinggi karena dokumennya sangat pendek dan hampir tanpa kata berulang, sehingga informasi frekuensi (yang dimanfaatkan TF-IDF tetapi diabaikan Jaccard) hampir tidak berpengaruh.

## 5. Pencarian Dokumen Berdasarkan Query (Top-K)

### 5.1 detik.com - query `hidup bahagia syukur`

In [ ]:
TopK = 5
query = "hidup bahagia syukur"

# Cosine similarity pada TF-IDF
query_tfidf = vec_tfidf_detik.transform([query])
similarities = cosine_similarity(X_tfidf_detik, query_tfidf)
top_indices = similarities[:, 0].argsort()[::-1][:TopK]

results_cosine = pd.DataFrame({
    "Teks": df_detik['prepro'].iloc[top_indices].values,
    "Cosine (TF-IDF)": similarities[top_indices, 0]
}, index=top_indices)
results_cosine

In [ ]:
# Jaccard similarity pada binary vector
query_bin = vec_bin_detik.transform([query]).toarray()
similarities_j = 1 - cdist(X_bin_detik, query_bin, metric='jaccard')
top_indices_j = similarities_j[:, 0].argsort()[::-1][:TopK]

results_jaccard = pd.DataFrame({
    "Teks": df_detik['prepro'].iloc[top_indices_j].values,
    "Jaccard": similarities_j[top_indices_j, 0]
}, index=top_indices_j)
results_jaccard

### 5.2 detik.com - query `gagal cerdas kerja`

In [ ]:
query = "gagal cerdas kerja"

query_tfidf = vec_tfidf_detik.transform([query])
similarities = cosine_similarity(X_tfidf_detik, query_tfidf)
top_indices = similarities[:, 0].argsort()[::-1][:TopK]

query_bin = vec_bin_detik.transform([query]).toarray()
similarities_j = 1 - cdist(X_bin_detik, query_bin, metric='jaccard')
top_indices_j = similarities_j[:, 0].argsort()[::-1][:TopK]

pd.DataFrame({
    "Teks (Cosine)": df_detik['prepro'].iloc[top_indices].values,
    "Cosine": similarities[top_indices, 0],
    "Teks (Jaccard)": df_detik['prepro'].iloc[top_indices_j].values,
    "Jaccard": similarities_j[top_indices_j, 0]
})

### 5.3 JSONPlaceholder - query `voluptatem omnis quia`

In [ ]:
query = "voluptatem omnis quia"

query_tfidf = vec_tfidf_api.transform([query])
similarities = cosine_similarity(X_tfidf_api, query_tfidf)
top_indices = similarities[:, 0].argsort()[::-1][:TopK]

query_bin = vec_bin_api.transform([query]).toarray()
similarities_j = 1 - cdist(X_bin_api, query_bin, metric='jaccard')
top_indices_j = similarities_j[:, 0].argsort()[::-1][:TopK]

pd.DataFrame({
    "Teks (Cosine)": df_api['prepro'].iloc[top_indices].str[:70].values,
    "Cosine": similarities[top_indices, 0],
    "Teks (Jaccard)": df_api['prepro'].iloc[top_indices_j].str[:70].values,
    "Jaccard": similarities_j[top_indices_j, 0]
})

In [ ]:
# berapa dokumen top-5 yang sama antara Cosine dan Jaccard?
len(set(top_indices) & set(top_indices_j))

### 5.4 JSONPlaceholder - query `et aut qui est`

In [ ]:
query = "et aut qui est"

query_tfidf = vec_tfidf_api.transform([query])
similarities = cosine_similarity(X_tfidf_api, query_tfidf)
top_indices = similarities[:, 0].argsort()[::-1][:TopK]

query_bin = vec_bin_api.transform([query]).toarray()
similarities_j = 1 - cdist(X_bin_api, query_bin, metric='jaccard')
top_indices_j = similarities_j[:, 0].argsort()[::-1][:TopK]

pd.DataFrame({
    "Teks (Cosine)": df_api['prepro'].iloc[top_indices].str[:70].values,
    "Cosine": similarities[top_indices, 0],
    "Teks (Jaccard)": df_api['prepro'].iloc[top_indices_j].str[:70].values,
    "Jaccard": similarities_j[top_indices_j, 0]
})

In [ ]:
len(set(top_indices) & set(top_indices_j))

### Analisis Pencarian Top-K

- **detik.com (`hidup bahagia syukur`):** kelima dokumen teratas Cosine dan Jaccard **sama (5 dari 5)**, dan dokumen teratas keduanya sama: `bijak bahagia syukur`. Perbedaannya ada pada urutan: dokumen `syukur syukur` berada di peringkat 2 menurut Cosine TF-IDF (sekitar 0.64) tetapi turun ke peringkat 4 menurut Jaccard (sekitar 0.33). Ini karena Cosine memperhitungkan frekuensi dan bobot kata (kata `syukur` muncul dua kali), sedangkan Jaccard menganggap dokumen itu hanya memuat satu kata `syukur`.
- **detik.com (`gagal cerdas kerja`):** hasilnya kembali konsisten (5 dari 5 sama) dan hanya 4 dokumen di seluruh korpus yang punya kata yang sama dengan query. Peringkat ke-5 sudah bernilai 0, yaitu dokumen yang sama sekali tidak relevan hanya "terpaksa" muncul karena kita meminta Top-5. Ini poin penting: **Top-K tidak menjamin hasilnya relevan, selalu cek nilai similarity-nya**.
- **JSONPlaceholder:** kesepakatan antar metode jauh lebih rendah, hanya **2 dari 5** dokumen yang sama untuk query `voluptatem omnis quia` dan **3 dari 5** untuk `et aut qui est`. Hampir seluruh dokumen (75 dan 98 dari 100) punya similarity di atas nol, dengan skor tertinggi yang rendah (cosine sekitar 0.32-0.46, Jaccard sekitar 0.12-0.17). Dokumen panjang dengan kosakata sempit membuat banyak dokumen terlihat "agak relevan", sehingga peringkat sangat sensitif terhadap metode yang dipakai. Karena teksnya *lorem ipsum* tanpa makna, peringkat ini juga tidak bisa dinilai benar/salah secara semantik.

## 6. Cosine Similarity pada Vektor Word2Vec & FastText

### 6.1 Tokenisasi & training model

In [ ]:
sentences_detik = [word_tokenize(teks.lower()) for teks in tqdm(df_detik['prepro'])]
sentences_api = [word_tokenize(teks.lower()) for teks in tqdm(df_api['prepro'])]

In [ ]:
# seed & workers=1 dipakai supaya hasil training bisa direproduksi
w2v_detik = gensim.models.Word2Vec(sentences_detik, min_count=1, vector_size=100, window=5, sg=0, seed=42, workers=1).wv
w2v_api = gensim.models.Word2Vec(sentences_api, min_count=1, vector_size=100, window=5, sg=0, seed=42, workers=1).wv

ft_detik = gensim.models.FastText(sentences_detik, min_count=1, vector_size=100, window=5,
                                  min_n=3, max_n=6, sg=1, epochs=10, seed=42, workers=1).wv
ft_api = gensim.models.FastText(sentences_api, min_count=1, vector_size=100, window=5,
                                min_n=3, max_n=6, sg=1, epochs=10, seed=42, workers=1).wv

### 6.2 Vektor dokumen (rata-rata vektor kata)

In [ ]:
def get_document_vector(tokens, wv):
    valid_vectors = [wv[word] for word in tokens if word in wv]
    if not valid_vectors:
        return np.zeros(wv.vector_size)
    return np.mean(valid_vectors, axis=0)

Xw2v_detik = np.array([get_document_vector(doc, w2v_detik) for doc in sentences_detik])
Xw2v_api = np.array([get_document_vector(doc, w2v_api) for doc in sentences_api])
Xft_detik = np.array([get_document_vector(doc, ft_detik) for doc in sentences_detik])
Xft_api = np.array([get_document_vector(doc, ft_api) for doc in sentences_api])

Xw2v_detik.shape, Xw2v_api.shape

### 6.3 Statistik cosine similarity antar dokumen

In [ ]:
sim_w2v_detik = cosine_similarity(Xw2v_detik)
sim_w2v_api = cosine_similarity(Xw2v_api)
sim_ft_detik = cosine_similarity(Xft_detik)
sim_ft_api = cosine_similarity(Xft_api)

pd.DataFrame({
    'Metode': ['TF-IDF', 'Word2Vec', 'FastText'],
    'Rata-rata detik': [sim_tfidf_detik[iu].mean(), sim_w2v_detik[iu].mean(), sim_ft_detik[iu].mean()],
    'Pasangan nol detik (%)': [(sim_tfidf_detik[iu] == 0).mean() * 100, (sim_w2v_detik[iu] == 0).mean() * 100, (sim_ft_detik[iu] == 0).mean() * 100],
    'Rata-rata jsonplaceholder': [sim_tfidf_api[iu_api].mean(), sim_w2v_api[iu_api].mean(), sim_ft_api[iu_api].mean()],
    'Pasangan nol jsonplaceholder (%)': [(sim_tfidf_api[iu_api] == 0).mean() * 100, (sim_w2v_api[iu_api] == 0).mean() * 100, (sim_ft_api[iu_api] == 0).mean() * 100],
}).round(3)

### 6.4 Query pada vektor embedding (dataset detik.com)

In [ ]:
query = "hidup bahagia syukur"
tokens_query = word_tokenize(query.lower())

hasil = {}
for nama, wv, X in [("Word2Vec", w2v_detik, Xw2v_detik), ("FastText", ft_detik, Xft_detik)]:
    q_vec = get_document_vector(tokens_query, wv).reshape(1, -1)
    sims = cosine_similarity(X, q_vec)[:, 0]
    top = sims.argsort()[::-1][:TopK]
    hasil[nama] = pd.DataFrame({"Teks": df_detik['prepro'].iloc[top].values, "Cosine": sims[top]}, index=top)

hasil["Word2Vec"]

In [ ]:
hasil["FastText"]

### 6.5 Query pada vektor embedding (dataset JSONPlaceholder)

In [ ]:
query = "voluptatem omnis quia"
tokens_query = word_tokenize(query.lower())

hasil_api = {}
for nama, wv, X in [("Word2Vec", w2v_api, Xw2v_api), ("FastText", ft_api, Xft_api)]:
    q_vec = get_document_vector(tokens_query, wv).reshape(1, -1)
    sims = cosine_similarity(X, q_vec)[:, 0]
    top = sims.argsort()[::-1][:TopK]
    hasil_api[nama] = pd.DataFrame({"Teks": df_api['prepro'].iloc[top].str[:70].values, "Cosine": sims[top]}, index=top)

hasil_api["Word2Vec"]

In [ ]:
hasil_api["FastText"]

### Analisis Cosine pada Word Embedding

Jalankan sel-sel di atas lalu bandingkan tabel 6.3 dengan hasil TF-IDF. Hal yang perlu diperhatikan:

- **Tidak ada pasangan bernilai nol.** Vektor embedding itu *dense* (semua dimensi berisi angka), jadi dua dokumen yang tidak punya kata yang sama pun tetap menghasilkan cosine tidak nol, berbeda dengan TF-IDF pada dataset detik yang sekitar 90% pasangannya nol. Nilai similarity rata-ratanya biasanya jauh lebih tinggi dan lebih "rapat" sehingga selisih antar dokumen kecil.
- **Embedding menangkap kemunculan bersama kata, bukan kecocokan kata persis.** Dua dokumen bisa dianggap mirip walau tidak berbagi kata, asalkan kata-katanya sering muncul pada konteks yang sama di korpus.
- **Keterbatasan:** korpus kita sangat kecil (58 dan 100 dokumen), jadi vektor yang dihasilkan masih *noisy* dan skor tinggi belum tentu berarti kemiripan makna yang nyata. Untuk dataset jsonplaceholder (teks acak) kemiripan makna memang tidak ada sejak awal, sehingga skor embeddingnya tidak bisa dibaca sebagai relevansi.
- Hasil training bisa sedikit berbeda antar-environment atau versi gensim walaupun `seed` sudah diset, jadi gunakan **angka dari output notebook kalian sendiri** saat menuliskan kesimpulan.

## 7. Ringkasan & Kesimpulan

In [ ]:
ringkasan = pd.DataFrame({
    'Metode': ['Cosine (BoW / TF-IDF)', 'Jaccard (One-Hot)', 'Cosine (Word2Vec / FastText)'],
    'Representasi': ['Sparse, berbobot frekuensi', 'Sparse, biner (ada/tidak)', 'Dense, vektor makna'],
    'Mempertimbangkan frekuensi kata?': ['Ya', 'Tidak', 'Tidak langsung'],
    'Bisa menilai dokumen tanpa kata yang sama?': ['Tidak', 'Tidak', 'Ya (berdasarkan konteks)'],
    'Sensitif terhadap panjang dokumen?': ['Tidak (dinormalisasi)', 'Ya (gabungan himpunan membesar)', 'Tidak (dirata-rata)'],
    'Butuh korpus besar?': ['Tidak', 'Tidak', 'Ya'],
})
ringkasan

**Kesimpulan:**

1. **Karakteristik data menentukan hasil similarity.** Dataset detik (dokumen sangat pendek) menghasilkan matriks similarity yang hampir kosong (sekitar 90% pasangan bernilai 0), sedangkan dataset jsonplaceholder (dokumen panjang, kosakata sempit) menghasilkan similarity yang rendah tetapi merata ke hampir semua pasangan.
2. **Cosine TF-IDF dan Jaccard konsisten untuk menentukan peringkat** (korelasi sekitar 0.94 di detik dan 0.87 di jsonplaceholder), tetapi Cosine lebih sensitif terhadap frekuensi dan kelangkaan kata, sedangkan Jaccard hanya melihat ada/tidaknya kata. Perbedaannya paling terlihat pada query dataset jsonplaceholder, di mana hanya 2-3 dari 5 dokumen teratas yang sama.
3. **Top-K selalu mengembalikan K dokumen walau tidak relevan**, sehingga nilai similarity harus selalu diperiksa (contoh query `gagal cerdas kerja` yang peringkat ke-5-nya bernilai 0).
4. **Cosine pada word embedding** bisa menilai kemiripan dokumen yang tidak berbagi kata, tetapi pada korpus sekecil ini hasilnya belum bisa dianggap reliabel secara semantik.
5. Sama seperti Exercise 5, dataset jsonplaceholder (*lorem ipsum*) menunjukkan bahwa **similarity tetap bisa dihitung secara matematis pada teks tanpa makna**, sehingga kebermaknaan data sumber harus dinilai terpisah dari angka similarity-nya.